# Superstore - Retail

In [1]:
import pandas as pd 
import numpy as np 

In [2]:
data = pd.read_csv("../data/raw/superstore-tableau.csv")

In [3]:
data.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,08-11-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,08-11-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,12-06-2016,16-06-2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,11-10-2015,18-10-2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,11-10-2015,18-10-2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [4]:
data.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='str')

In [5]:
# Creating unique data tables

customer = data[["Customer ID", "Customer Name", "Segment", "Country", "State", "City", "Postal Code", "Region"]].drop_duplicates()
product = data[["Product ID", "Category", "Sub-Category", "Product Name"]].drop_duplicates()
orders = data[["Customer ID", "Order ID", "Order Date", "Ship Date", "Ship Mode", "Product ID", "Sales", "Quantity", "Discount", "Profit"]].drop_duplicates()
orders['Order Date'] = pd.to_datetime(orders['Order Date'], format = "mixed")
orders['Ship Date'] = pd.to_datetime(orders['Ship Date'], format = "mixed")


In [6]:
dates = pd.DataFrame(orders['Order Date'].unique()).sort_values(by=0).reset_index(drop=True)
dates.columns = ['date']
dates['month'] = dates['date'].dt.month_name()
dates['month_num'] = dates['date'].dt.month
dates['year'] = dates['date'].dt.year


In [7]:
# Let's check total  orders
print("Total orders:", len(orders['Order ID'].unique()))

# Let's check total  sales
print("Total sales:", round(orders['Sales'].sum(), 2))

# Let's check total  customers
print("Total customers:", len(orders['Customer ID'].unique()))


Total orders: 5009
Total sales: 2296919.49
Total customers: 793


In [8]:
# Total Sales Year-on-Year
sales_yoy = pd.DataFrame(orders.merge(dates[['date','year']], left_on="Order Date", right_on = "date").groupby(['year'])['Sales'].sum())
x1 = sales_yoy.Sales.values[0]
cum_sales = []
for x in sales_yoy.Sales.values:
    if(x==x1):
        cum_sales.append(0)
    else:
        cum_sales.append(round((x-x1)/x,2))
        x1 = x
sales_yoy['%_change_in_sales'] = cum_sales
sales_yoy

,Sales,%_change_in_sales
year,,
2014,483966.1261,0.00
2015,470532.5090,-0.03
2016,609205.5980,0.23
2017,733215.2552,0.17


In [26]:
# Monthly sales across years 2015-2017
sales_2015 = round(pd.DataFrame(orders.merge(dates[dates['year'] == 2015][['date','month_num']], 
                                      left_on="Order Date", 
                                     right_on = "date").groupby(['month_num'])['Sales'].sum()).sort_values('month_num'),2)
sales_2016 = round(pd.DataFrame(orders.merge(dates[dates['year'] == 2016][['date','month_num']], 
                                      left_on="Order Date", 
                                     right_on = "date").groupby(['month_num'])['Sales'].sum()).sort_values('month_num'),2)
sales_2017 = round(pd.DataFrame(orders.merge(dates[dates['year'] == 2017][['date','month_num']], 
                                      left_on="Order Date", 
                                     right_on = "date").groupby(['month_num'])['Sales'].sum()).sort_values('month_num'),2)
monthly_sales_all = (sales_2015.merge(sales_2016, "left", on="month_num")).merge(sales_2017, "left", on="month_num")
monthly_sales_all.columns = ["Sales_2015","Sales_2016", "Sales_2017"]
monthly_sales_all = monthly_sales_all.merge(dates[['month_num',"month"]].drop_duplicates(), "left", on ="month_num")[["month","Sales_2015","Sales_2016","Sales_2017"]]
monthly_sales_all

,month,Sales_2015,Sales_2016,Sales_2017
0,January,29347.39,38048.18,64734.31
1,February,20728.35,49238.41,50011.49
2,March,40876.61,49612.04,74774.08
3,April,38056.97,45192.28,39072.00
4,May,30933.71,64964.32,40882.45
5,June,28862.20,38991.94,47742.33
6,July,28730.38,42773.40,54382.09
7,August,50094.53,46339.99,75675.30
8,September,66729.33,41985.14,74164.61
9,October,32025.08,52268.15,65501.16


In [ ]:
# % change in monthly sales across years 2015-2017

In [ ]:
# Total Sales Month-on-Month latest for year 2017
sales_yoy_2017 = round(pd.DataFrame(orders.merge(dates[dates['year'] == 2017][['date','year','month_num', 'month']], 
                                      left_on="Order Date", 
                                     right_on = "date").groupby(['month_num', 'month'])['Sales'].sum()).sort_values('month_num'),2)
# % of change in sales
x1 = sales_yoy_2017.Sales.values[0]
cum_sales = []
for x in sales_yoy_2017.Sales.values:
    if(x==x1):
        cum_sales.append(0)
    else:
        cum_sales.append(round((x-x1)/x,2))
        x1 = x
sales_yoy_2017['%_change_in_sales'] = cum_sales


# calculate cumulative sales
x1 = sales_yoy_2017.Sales.values[0]
cum_sales_month = []
for x in sales_yoy_2017.Sales.values:
    if(x==x1):
        cum_sales_month.append(x)
    else:
        cum_sales_month.append(round((x+x1),2))
        x1 = x+x1
sales_yoy_2017['Cumulative_Sales'] = cum_sales_month
sales_yoy_2017
x1 = sales_yoy_2017.Sales.values[0]


sales_yoy_2017

In [ ]:
sales_dec_17 = pd.DataFrame(orders.merge(dates[dates['year'] == 2017][dates['month'] == 'December'].reset_index(drop=True), 
                                    "inner",
                                      left_on="Order Date", 
                                      right_on = "date"))

In [ ]:
sales_dec_17

In [ ]:
sales_dec_17[['Product ID', 'Sales', 'Quantity', 'Profit']].sort_values()